## Concept 8 — Contextual Compression

### What is it?
Even after good retrieval, each chunk contains irrelevant sentences mixed in with the useful ones.
Contextual Compression **extracts only the relevant portion** of each doc before passing to the LLM.

### Pipeline
```
Query
  → Retrieve docs (full chunks)
  → Compressor: extract only sentences relevant to the query from each doc
  → LLM gets clean, focused context
  → LLM → Answer
```

### Benefits
- LLM focuses on relevant text only → better answers
- Fewer tokens sent to LLM → cheaper
- More docs can fit in context window

### LangChain has this built-in
`ContextualCompressionRetriever` wraps any retriever and adds compression automatically.
`LLMChainExtractor` uses the LLM to extract relevant sentences.

### Limitation (what Concept 9 fixes)
Even after compression, if your knowledge base simply doesn't have the answer, the system still tries to answer with what it has.
→ Concept 9 (CRAG) evaluates doc quality and falls back to web search when the knowledge base can't help.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

chunks, vectorstore, embeddings, llm, prompt = setup()

### Step 1 — Create Base Retriever

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Step 2 — See a Raw Retrieved Doc (Before Compression)
Notice how much irrelevant text is in each chunk.

In [ ]:
query    = TEST_QUERIES[1]  # 'What are checkpoints in LangSmith?'
raw_docs = base_retriever.invoke(query)

print(f"Query: {query}")
print(f"\n--- Raw Doc 1 (full chunk, {len(raw_docs[0].page_content)} chars) ---")
print(raw_docs[0].page_content)

### Step 3 — Create Compressor
`LLMChainExtractor` asks the LLM to extract only the sentences relevant to the query.

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

### Step 4 — See Compressed Doc (After Compression)
Compare the compressed output to the raw doc above.

In [ ]:
compressed_docs = compression_retriever.invoke(query)

print(f"Docs after compression: {len(compressed_docs)}")
print(f"\n--- Compressed Doc 1 ({len(compressed_docs[0].page_content) if compressed_docs else 0} chars) ---")
if compressed_docs:
    print(compressed_docs[0].page_content)

### Step 5 — Full Contextual Compression Function

In [ ]:
def contextual_compression_rag(query):
    # compression_retriever handles retrieval + compression in one step
    compressed_docs = compression_retriever.invoke(query)

    if not compressed_docs:
        return "I don't know — no relevant content found after compression."

    context  = format_docs(compressed_docs)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [ ]:
run_queries(contextual_compression_rag, label="Concept 8 — Contextual Compression")